# 12 · Inundación con Sentinel-1 SAR

**Objetivo:** Detectar expansión de superficies de baja retrodispersión durante un evento.

**Datos:** COPERNICUS/S1_GRD.

**Relevancia para política ambiental y social:** Permite análisis aun con nubosidad y apoya respuesta humanitaria.

**Limitaciones:** Sombras radar y superficies lisas pueden confundirse con agua.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def s1(start,end):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi).filterDate(start,end)
        .filter(ee.Filter.eq("instrumentMode","IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VV"))
        .select("VV").median().clip(aoi)
    )
before = s1("2025-01-01","2025-01-31")
during = s1("2025-02-01","2025-02-28")
change = during.subtract(before)
flood = change.lt(-3).selfMask()
Map.addLayer(change, {"min":-6,"max":6,"palette":["blue","white","red"]}, "Cambio VV", False)
Map.addLayer(flood, {"palette":["00FFFF"]}, "Inundación probable")
Map
